In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
df = pd.read_csv("../data/ai4i2020_augmented.csv")

df = df.rename(columns={
    "Quality Type": "quality_type",
    "Machine Type": "machine_type",
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Cutting force [N]": "cutting_force",
    "Tool flank wear VB [mm]": "flank_wear",
    "Machine failure": "failure"
})

In [3]:
df = df.drop(columns=["UDI", "Product ID"])
df.columns

Index(['quality_type', 'machine_type', 'air_temp', 'process_temp', 'rpm',
       'torque', 'cutting_force', 'flank_wear', 'failure', 'TWF', 'HDF', 'PWF',
       'OSF', 'RNF'],
      dtype='str')

In [5]:
df["quality_type"] = df["quality_type"].map({"L": 0, "M": 1, "H": 2})
df["machine_type"] = df["machine_type"].map({"Milling": 0, "Turning": 1})

df[["quality_type", "machine_type"]].isnull().sum()

quality_type    0
machine_type    0
dtype: int64

In [6]:
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"] * 2 * np.pi / 60
df["wear_force"] = df["flank_wear"] * df["cutting_force"]

In [7]:
df[["temp_diff", "power", "wear_force"]].describe()

,temp_diff,power,wear_force
count,10000.000000,10000.000000,10000.000000
mean,10.000630,6279.744953,351.659653
std,1.001094,1067.418295,292.792945
min,7.600000,1148.440610,2.750000
25%,9.300000,5561.184484,122.702025
50%,9.800000,6271.027344,277.831500
75%,11.000000,7003.002724,497.086000
max,12.100000,10469.923005,2072.667200


In [8]:
failure_types = df[["TWF", "HDF", "PWF", "OSF", "RNF"]]

In [9]:
X = df.drop(columns=["failure", "TWF", "HDF", "PWF", "OSF", "RNF"])
y = df["failure"]

print(X.columns.tolist())
print(X.shape, y.shape)

['quality_type', 'machine_type', 'air_temp', 'process_temp', 'rpm', 'torque', 'cutting_force', 'flank_wear', 'temp_diff', 'power', 'wear_force']
(10000, 11) (10000,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean() * 100, y_test.mean() * 100)

(8000, 11) (2000, 11)
3.55 3.55


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

X_train_scaled.head()

,quality_type,machine_type,air_temp,process_temp,rpm,torque,cutting_force,flank_wear,temp_diff,power,wear_force
4011,-0.745335,-0.804641,1.098847,0.738874,-0.513337,1.218880,1.918680,1.169663,-1.100873,1.515020,2.257930
4360,-0.745335,-0.804641,1.048764,0.062737,0.021382,-0.418880,0.215225,0.869216,-2.000555,-0.463252,0.712375
7812,-0.745335,1.242791,-0.002980,1.009328,0.706490,-0.800688,-1.144874,0.268321,1.498208,-0.652485,-0.459884
7207,0.744963,-0.804641,-0.153229,-0.275331,0.004672,-0.559546,0.063710,-0.257462,-0.101227,-0.685474,-0.216606
7409,-0.745335,-0.804641,-0.103146,0.603646,3.034743,-2.247543,-1.158658,1.137472,1.098349,-2.285583,-0.126499


In [13]:
import os
os.makedirs("../data/processed", exist_ok=True)   # use "data/processed" if needed

X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# failure types for Module 6 (same rows as train and test)
failure_types.loc[X_train.index].to_csv("../data/processed/types_train.csv", index=False)
failure_types.loc[X_test.index].to_csv("../data/processed/types_test.csv", index=False)

joblib.dump(scaler, "../models/scaler.joblib")

['../models/scaler.joblib']